In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

import numpy as np 
import pandas as pd 
#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

# import stata_setup
# stata_setup.config('C:/Users/Farid Srouji/Stata15/', 'mp')
# import sys
# sys.path.append("C:/Users/Farid Srouji/Stata15/utilities")
# from pystata import config
# config.init()

In [ ]:
# Import packages
import os
import pickle

import numpy as np 
import pandas as pd 

import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

import tensorflow as tf
from tensorflow import keras

from scipy import stats
import scipy.optimize as optimize 

In [ ]:
# Reading the saved data pickle file
df_vix = pd.read_pickle('/Users/Farid Srouji/Documents/Research/VIX prediction/master/financialmodelingprep/data/pkl/pickled_filtered_lead_para_sp500_clean.pkl', compression=None)

In [ ]:
df_vix

In [ ]:
df_vix[["VIX"]] = df_vix["adj close"].apply(np.int)

In [ ]:
df_vix

In [ ]:
#df_vix = df_vix.reset_index()

In [ ]:
#df_vix['index']

In [ ]:
#df_vix = df_vix.reset_index()
# df_vix['adj close'] = df_vix['adj close'].apply(np.float64) + 0.5
#df_vix['VIX'] = df_vix['adj close']
#df_vix.columns.values[1] = "VIX"
#decimals = 2
#df_vix['VIX'] = df_vix['adj close'].apple(np.round)
#df_vix['text'] = df_vix['articles'].str()

In [ ]:
#df_vix[['index','date']] = df_vix['index'].split(' ',expand=True)
#tmpDF = pd.DataFrame(columns=['index','date'])
#df_vix[['index','date']] = df_vix[['index']].str.split(' ', expand=True)

In [ ]:
df_vix



In [ ]:
# selecting the VIX and articles
df_vix = df_vix[['VIX', 'articles']]

In [ ]:
#df_vix['articles'] = df_vix['articles'].map(lambda x: x.lstrip('.-'))

In [ ]:
df_vix

In [ ]:
df = df_vix[['VIX']].copy()
df


In [ ]:
df_vix = df_vix.reindex(np.random.permutation(df_vix.index))  
df_vix = df_vix[['VIX', 'articles']]

In [ ]:
# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['sentiment']] = ''

In [ ]:
df

In [ ]:
#import nltk
#nltk.download('vader_lexicon')

In [ ]:
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata
sid = SentimentIntensityAnalyzer()
countNeg=0
countNeu=0
countPos=0
#print(df_vix['articles'])
countNegWord=0
countNeuWord=0
countPosWord=0
for date, row in df_vix.T.iteritems():
    print("The date is:"+str(date))
    try:
        sentence = unicodedata.normalize('NFKD', df_vix.loc[date, 'articles'])#.encode('ascii','ignore').decode('ascii')
        #sentence = unicodedata.normalize('NFKD', df_vix.loc[date, 'articles']).encode('ascii','ignore')
        # add my own sentence Options sentiment.
        #print("#1")
        #print (df_vix.loc[date, 'articles'])
        ss = sid.polarity_scores(sentence)
        #print(sentence)
        #print("#2")
        df.at[date, 'compound'] = ss['compound']
        df.at[date, 'neg'] = ss['neg']
        df.at[date, 'neu'] = ss['neu']
        df.at[date, 'pos'] = ss['pos']
        if ((ss['pos'] + ss['neg']) != 0):
            number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
            if (number != ''):
                df.at[date, 'sentiment'] = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        countNeg+=ss['neg']
        countNeu+=ss['neu']
        countPos+=ss['pos']
        
        print("============sentiment status============")
        print("compound="+str(df.at[date, 'compound']))
        print("neg="+str(df.at[date, 'neg']))
        print("neu="+str(df.at[date, 'neu']))
        print("pos="+str(df.at[date, 'pos']))
        print("sentiment="+str(df.at[date, 'sentiment']))
        #print("Sentence="+sentence)
        words = sentence.split()
        print("----------start sentenct-------")

        for word in words:
            try:
                #print("word="+word)
                ww = sid.polarity_scores(word)
                if (ww['neg']==1):
                    countNegWord+=1
                elif (ww['neu']==1):
                    countNeuWord+=1
                elif (ww['pos']==1):
                    countPosWord+=1
                else :
                    print("Error ww="+ww)
            except:
                print("Exception for word="+word)

        print("----------end sentenct-------")

        #print("passed")
    except TypeError:
        #print("sentence is")
        #print(sentence)
        #print(df_vix.loc[date, 'articles'])
        print("ERROR")
        df.at[date, 'compound'] = 0
        df.at[date, 'neg'] = 0
        df.at[date, 'neu'] = 0
        df.at[date, 'pos'] = 0

In [ ]:
#Data Preprocessing for Neural Net
#In this section, we preprocess data before feeding into neural network.

In [ ]:
train_start_date = '2019-09-25'
train_end_date = '2021-06-24'
test_start_date = '2019-09-25'
test_end_date = '2021-06-24'

train_org = df.loc[train_start_date : train_end_date]
print(test_start_date)
test_org = df.loc[test_start_date:test_end_date]
print(test_end_date)


In [ ]:
#1. Normalize data

In [ ]:
# 1. Normalize data

# Keep the original
train = train_org.copy()
test = test_org.copy()

# Variables that has to be normalized
to_normalize = ['VIX','sentiment']

# Normalize
mean_train = train[to_normalize].mean()
std_train = train[to_normalize].std()

train[to_normalize] = (train[to_normalize]-mean_train)/std_train
test[to_normalize] = (test[to_normalize]-mean_train)/std_train

In [ ]:
#2. Select input variables

In [ ]:
# Select input variables
input_var = ['VIX', 'sentiment'] 
print(f'total {len(input_var)} variables')

# Subsample data
x_train_org = train[input_var]
x_test_org = test[input_var]
y_train_org = train['VIX']
y_test_org = test['VIX']

print('x_train_org', x_train_org.shape)
print('y_train_org', y_train_org.shape)
print('x_test_org', x_test_org.shape)
print('y_test_org', y_test_org.shape)

In [ ]:
# 3. Reshape data¶
# We set our time steps to be 4.
# This means that we predict the VKOSPI with previous data from last 5 days.
# It's 5, not 4, because we already shifted 1 day before EDA (in 1-3).
# 5 days usually represent one week. (excluding weekend)
# Shape of reshpaed data is (batch_size, time_steps, number_of_feautures).

# number of features will remain the same.
# time steps will be set to 4.

In [ ]:
TIME_STEPS = 4  #predict with 5 days (+1 because we already shifted 1 day in 1-3)

# function
def reshape(data, time_steps = TIME_STEPS):
    stack = [data.iloc[i:i+time_steps].to_numpy() for i in range(len(data)-time_steps)]
    reshaped = np.stack(stack, axis=0) 
    return reshaped

# reshape
x_train_val = reshape(x_train_org) # train data and validation data
y_train_val = reshape(y_train_org)
x_test = reshape(x_test_org)
y_test = reshape(y_test_org)

print('x_train + x_val', x_train_val.shape)
print('y_train + y_val', y_train_val.shape)
print('x_test', x_test.shape)
print('y_test', y_test.shape)

In [ ]:
#4. Seperate validation data from train data

In [ ]:
# Seperate validation data from train data
i = int(len(x_train_val) * 0.8)

x_train = x_train_val[:i]
x_val = x_train_val[i:]
y_train = y_train_val[:i]
y_val = y_train_val[i:]

print('train data:', x_train.shape[0])
print('validation data:', x_val.shape[0])
print('test data:', x_test.shape[0])

In [ ]:
# Hyperparameter tuning
# Bayesian optimization will be applied using kerastuner.

In [ ]:
from kerastuner import BayesianOptimization
# Function 'model_builder' builds and compiles neural network with given hyperparameters

# vanilla nn
def nn_builder(hp):
    # hp
    UNITS = hp.Int('UNITS', min_value = 16, max_value = 128, step = 16)
    ACTIVATION_1 = hp.Choice('ACTIVATION',values = ['relu','linear','tanh'])
    ACTIVATION_2 = hp.Choice('ACTIVATION',values = ['relu','linear','tanh'])
    
    # model instance
    model = keras.Sequential([
        keras.layers.Flatten(),
        keras.layers.Dense(units = UNITS, activation = ACTIVATION_1),
        keras.layers.Dropout(rate = 0.2),
        keras.layers.Dense(1, activation = ACTIVATION_2)
    ])
    
    # compile
    model.compile(optimizer = keras.optimizers.Adam(),
                  loss = 'mse', metrics = ['mae'])    
    
    return model

# lstm
def lstm_builder(hp):
    # hp
    UNITS_1 = hp.Int('UNITS_1', min_value = 16, max_value = 128, step = 16)
    UNITS_2 = hp.Int('UNITS_2', min_value = 8, max_value = 32, step = 8)
    ACTIVATION = hp.Choice('ACTIVATION',values = ['relu','linear','tanh'])
    
    # model instance
    model = keras.Sequential([
        # input_shape = (time_steps, # of features)
        keras.layers.LSTM(units = UNITS_1, input_shape = (None, x_train.shape[2]), return_sequences = False), 
        keras.layers.Dense(UNITS_2, activation = ACTIVATION),
        keras.layers.Dropout(rate = 0.2),
        keras.layers.Dense(1, activation = 'linear')
    ])
    
    # compile
    model.compile(optimizer = keras.optimizers.Adam(),
                  loss = 'mse', metrics = ['mae'])
    
    return model

# gru
def gru_builder(hp):
    # hp
    UNITS_1 = hp.Int('UNITS_1', min_value = 16, max_value = 128, step = 16)
    UNITS_2 = hp.Int('UNITS_2', min_value = 8, max_value = 32, step = 8)
    ACTIVATION = hp.Choice('ACTIVATION',values = ['relu','softmax','linear'])
    
    # model instance
    model = keras.Sequential([
        # input_shape = (time_steps, # of features)
        keras.layers.GRU(units = UNITS_1, input_shape = (None, x_train.shape[2]), return_sequences = False),
        keras.layers.Dense(UNITS_2, activation = ACTIVATION),
        keras.layers.Dropout(rate = 0.2),
        keras.layers.Dense(1, activation = 'linear')
    ])
    
    # compile
    model.compile(optimizer = keras.optimizers.Adam(),
                  loss = 'mse', metrics = ['mae'])
    
    return model

In [ ]:
# Implement earlystopping with keras callback 
from tensorflow import keras

PATIENCE = 5 # number of epochs with no improvement after which training will be stopped.

Earlystopping = keras.callbacks.EarlyStopping(monitor='val_loss',
                                              min_delta = 0.001,
                                              patience=PATIENCE, 
                                              mode='min', 
                                              restore_best_weights=True)

In [ ]:
# Search hyperparameters
SEED = 121

# NN
tuner_nn = BayesianOptimization(nn_builder,
                                objective = 'val_loss',
                                max_trials = 20,
                                seed = SEED,
                                directory = 'kerastuner',
                                overwrite = True
                                )
x_train = np.asarray(x_train).astype('float32')
y_train = np.asarray(y_train).astype('float32')
x_val = np.asarray(x_val).astype('float32')
y_val = np.asarray(y_val).astype('float32')
#show_shapes()

tuner_nn.search(x_train, y_train, epochs=50, validation_data=(x_val, y_val), verbose=0, callbacks=[Earlystopping])

## Build model based on the optimized hyperparameters
besthp_nn = tuner_nn.get_best_hyperparameters()[0]
model_nn = tuner_nn.hypermodel.build(besthp_nn)


# # lstm
# tuner_lstm = BayesianOptimization(lstm_builder,
#                             objective = 'val_loss',
#                             max_trials = 20,
#                             seed = SEED,
#                             directory = 'kerastuner')

# tuner_lstm.search(x_train, y_train, epochs=50, validation_data=(x_val, y_val), verbose=0, callbacks=[Earlystopping])

# ## Build model based on the optimized hyperparameters
# besthp_lstm = tuner_lstm.get_best_hyperparameters()[0]
# model_lstm = tuner_lstm.hypermodel.build(besthp_lstm)


# gru
tuner_gru = BayesianOptimization(gru_builder,
                            objective = 'val_loss',
                            max_trials = 20,
                            seed = SEED,
                            directory = 'kerastuner')

tuner_gru.search(x_train, y_train, epochs=50, validation_data=(x_val, y_val), verbose=0, callbacks=[Earlystopping])

# ## Build model based on the optimized hyperparameters
# besthp_gru = tuner_gru.get_best_hyperparameters()[0]
# model_gru = tuner_gru.hypermodel.build(besthp_gru)

In [ ]:
besthp_nn

In [ ]:
# Colors designated to each model
palette = default_clrs[:4]
palette.append(default_clrs[6])